<img src="../pic/CU.png" style="width: 100%; display: block; margin: auto;">

# Курс по прогнозированию временных рядов
# Неделя 4. Семинар 2.
# Автоподбор порядка

> 🔑 **Главная мысль ноутбука.** `AutoARIMA` — это не оракул, а короткая
> эвристика: выбрать $d$ тестом, потом жадно спускаться по сетке $(p, q)$ в
> сторону меньшего AICc. Она быстрая, обычно хорошая и **не гарантирует**
> оптимума. Понимая её устройство, вы знаете, когда ей можно верить, а когда
> нужно перебрать руками.

## Что мы разберём

| Раздел | Вопрос, на который отвечаем |
|---|---|
| 0 | Данные и инструменты ноутбука |
| 1 | Что на самом деле делает `auto.arima` — алгоритм Хайндмана–Кхандакара |
| 2 | Пишем этот перебор сами и сверяем с библиотекой |
| 3 | AICc против кросс-валидации: два разных вопроса к одной модели |
| 4 | Когда все кандидаты проваливают тесты — и почему это не повод не прогнозировать |


In [ ]:
import time
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from statsmodels.tsa.stattools import kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

import statsforecast
from statsforecast.models import AutoARIMA, AutoETS
from statsforecast.arima import arima_string

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["figure.dpi"] = 100

print("statsmodels:   ", sm.__version__)
print("statsforecast: ", statsforecast.__version__)

---
# 0. Данные и инструменты

## 0.1. Сквозные функции семинара

`tsdisplay` и `aicc` — те же, что и в предыдущих ноутбуках. Приводим их ещё
раз, чтобы ноутбук запускался сам по себе.

In [ ]:
def tsdisplay(y, lags=36, title=None, figsize=(13, 6)):
    """Панель «ряд + ACF + PACF» — сквозной инструмент диагностики семинара."""
    y = pd.Series(y).dropna()
    lags = min(lags, len(y) // 2 - 1)

    fig = plt.figure(figsize=figsize)
    grid = fig.add_gridspec(2, 2)
    ax_series = fig.add_subplot(grid[0, :])
    ax_acf = fig.add_subplot(grid[1, 0])
    ax_pacf = fig.add_subplot(grid[1, 1])

    ax_series.plot(y.index, y.to_numpy())
    ax_series.set_title(title or "Временной ряд")

    plot_acf(y, lags=lags, ax=ax_acf, title="ACF")
    plot_pacf(y, lags=lags, ax=ax_pacf, method="ywm", title="PACF")

    fig.tight_layout()
    return fig


def aicc(res):
    """AIC с поправкой на конечную выборку (ноутбук `03`, раздел 0.2)."""
    k = len(res.params)      # phi, theta, константа (если есть) и sigma^2
    n = res.nobs             # число наблюдений, вошедших в правдоподобие
    if n - k - 1 <= 0:
        return np.inf
    return res.aic + 2 * k * (k + 1) / (n - k - 1)


def ets_params(res):
    """Параметры ETSModel в виде обычного словаря.

    🔍 У `ETSResults` атрибут `.params` — это numpy-массив без имён
    (в отличие от `ARIMAResults`, где он проиндексирован). Имена лежат
    отдельно, в `.param_names`, — их и склеиваем.
    """
    return dict(zip(res.param_names, np.asarray(res.params)))

## 0.2. Данные

Три ряда, все уже знакомые:

* **US consumption** (`us_change.csv`) — квартальный прирост потребительских
  расходов в США, 187 наблюдений. Сквозной ряд семинара: стационарный, $d = 0$;
* **поголовье скота в Азии** (`livestock.csv`) — годовой ряд из 47 точек с
  почти линейным трендом и без сезонности. Идеальный полигон для сравнения
  ARIMA и ETS: обе модели умеют такие ряды, и обе — по-своему;
* **a10** (`a10.csv`) — месячные расходы на противодиабетические препараты в
  Австралии. Ряд с трендом **и с сильной годовой сезонностью**. Сегодня он
  нужен именно как «неудобный» пример: несезонная ARIMA его в принципе не
  может описать до конца, и мы посмотрим, что с этим делать (раздел 4).

In [ ]:
us = pd.read_csv("../../4/data/us_change.csv", parse_dates=["Time"]).set_index("Time")
us.index.freq = "QS"
consumption = us["Consumption"]

_liv = pd.read_csv("../../4/data/livestock.csv")
livestock = pd.Series(
    _liv["value"].to_numpy(),
    index=pd.PeriodIndex(_liv["time"].astype(int), freq="Y").to_timestamp(),
    name="livestock",
)
livestock.index.freq = "YS-JAN"

a10 = pd.read_csv("../../4/data/a10.csv", parse_dates=["Month"]).set_index("Month")["Cost"]
a10.index.freq = "MS"

fig, axes = plt.subplots(1, 3, figsize=(14, 3.2))
consumption.plot(ax=axes[0], title=f"US consumption ({len(consumption)} кв.)")
livestock.plot(ax=axes[1], title=f"Поголовье скота ({len(livestock)} лет)")
a10.plot(ax=axes[2], title=f"a10 ({len(a10)} мес.)")
fig.tight_layout()
plt.show()

---
# 1. Что на самом деле делает `auto.arima`

## 1.1. Три вопроса, а не один

Когда мы пишем `AutoARIMA(...).fit(y)`, автомат отвечает **на три разных
вопроса**, и отвечает на них **разными способами**:

| Вопрос | Чем решается | Почему именно так |
|---|---|---|
| Сколько раз дифференцировать, $d$? | тест на единичный корень (по умолчанию KPSS) | AICc **не умеет** сравнивать модели с разным $d$ (ноутбук `03`, раздел 5.4) |
| Какие $p$ и $q$? | минимум AICc по сетке | это и есть задача выбора модели при фиксированных данных |
| Нужна ли константа/снос? | тоже минимум AICc | это ещё один бинарный параметр модели |

🔑 **Первое, что нужно понять про автоподбор: он не перебирает $(p, d, q)$ по
одному критерию.** $d$ выбирается **до** и **вне** перебора. Это прямое
следствие того же ограничения, которое мы обсуждали в ноутбуке `03`:
при $d = 0$ и $d = 1$ правдоподобия считаются по разным наборам данных, и
сравнивать их бессмысленно.

## 1.2. Почему не полный перебор

Казалось бы, чего мудрить: переберём все $(p, q)$ до пятого порядка и возьмём
минимум. Посчитаем, сколько это моделей.

In [ ]:
rows = []
for max_pq in (2, 3, 5, 8):
    n_nonseasonal = (max_pq + 1) ** 2 * 2                       # (p, q) × {с константой, без}
    n_seasonal = (max_pq + 1) ** 2 * (2 + 1) ** 2 * 2           # ещё и (P, Q) до 2
    rows.append({"max p, q": max_pq,
                 "несезонных моделей": n_nonseasonal,
                 "сезонных моделей (P, Q ≤ 2)": n_seasonal})
pd.DataFrame(rows)

Для несезонного ряда полный перебор — это десятки моделей, вполне подъёмно.
А вот для сезонного — уже сотни, и каждая подгонка стоит недёшево
(в ноутбуке `04` мы видели: одна итерация фильтра Калмана — $O(n r^2)$,
а итераций у оптимизатора десятки).

Отсюда конструкция алгоритма: **не обходить сетку целиком, а спускаться по
ней жадно**, проверяя только соседей текущей лучшей модели.

## 1.3. Алгоритм Хайндмана–Кхандакара

Тот самый алгоритм, который стоит за `auto.arima` в R и за `AutoARIMA` в
`statsforecast` (Hyndman & Khandakar, *Journal of Statistical Software*, 2008).
В несезонном варианте он выглядит так.

> **Шаг 0.** Выбрать $d$ по повторяющемуся KPSS-тесту: пока тест отвергает
> стационарность — брать разность (это функция `ndiffs` из ноутбука `1.03`).
>
> **Шаг 1.** Подогнать **четыре стартовые модели**:
>
> * ARIMA(2, $d$, 2),
> * ARIMA(0, $d$, 0),
> * ARIMA(1, $d$, 0),
> * ARIMA(0, $d$, 1).
>
> Константа включается, если $d \le 1$. Лучшую из четырёх по AICc объявить
> **текущей**.
>
> **Шаг 2.** Рассмотреть **вариации текущей модели**:
>
> * $p$ изменить на $\pm 1$;
> * $q$ изменить на $\pm 1$;
> * $p$ и $q$ одновременно на $\pm 1$;
> * включить или выключить константу.
>
> Если какая-то вариация дала меньший AICc — она становится текущей, и
> шаг 2 повторяется. Если ни одна не улучшила AICc — останов.

## 1.4. Почему стартовые модели именно такие

Четвёрка из шага 1 подобрана не случайно — это четыре «угла» пространства
моделей:

| Модель | Что она проверяет |
|---|---|
| ARIMA(0, $d$, 0) | «а может, после разностей уже белый шум?» — самая простая гипотеза |
| ARIMA(1, $d$, 0) | чистая AR минимального порядка |
| ARIMA(0, $d$, 1) | чистая MA минимального порядка |
| ARIMA(2, $d$, 2) | середина пространства — точка, из которой можно спуститься в любую сторону |

🔍 Смысл ARIMA(2, $d$, 2) — **не** в том, что это хорошая модель, а в том, что
из неё шаг 2 может уйти и вверх, и вниз по обоим индексам. Стартуй алгоритм
только из нуля, он никогда бы не добрался до моделей, у которых улучшение
начинается лишь с порядка 2 (одиночный шаг из (0,$d$,0) в (1,$d$,0) может AICc
ухудшить, хотя (2,$d$,2) была бы лучше всех).

## 1.5. Ограничители, о которых стоит знать

У `AutoARIMA` в `statsforecast` есть параметры, напрямую следующие из
устройства алгоритма:

In [ ]:
print("Значения по умолчанию у AutoARIMA (несезонный случай):\n")
for name, value, comment in [
    ("d", None, "если None — выбирается тестом (шаг 0)"),
    ("test", "'kpss'", "какой тест используется для выбора d"),
    ("ic", "'aicc'", "критерий сравнения; в лекции рекомендован именно AICc"),
    ("start_p, start_q", "2, 2", "координаты стартовой модели шага 1"),
    ("max_p, max_q", "5, 5", "границы сетки"),
    ("max_order", 5, "ограничение на p + q: слишком богатые модели не рассматриваются"),
    ("stepwise", True, "True — обход шага 2; False — полный перебор"),
    ("nmodels", 94, "предохранитель: сколько моделей максимум успеет посмотреть обход"),
    ("approximation", False, "True — считать приближённое правдоподобие (быстро, грубо)"),
    ("allowdrift, allowmean", "True, True", "разрешено ли включать снос / константу"),
    ("trace", False, "печатать каждую посещённую модель"),
]:
    print(f"  {name:<22} = {str(value):<12} # {comment}")

🔍 **`max_order=5` — самый неочевидный из них.** Он ограничивает не $p$ и не
$q$ по отдельности, а их **сумму**. То есть ARIMA(4,$d$,4) при настройках по
умолчанию не будет рассмотрена никогда, даже при `stepwise=False`, потому что
$4 + 4 = 8 > 5$. Если вам нужна такая модель — поднимайте `max_order` явно.

🔍 **`approximation`** включается автоматически на длинных рядах (в R —
при $n > 150$ или при большом сезонном периоде): вместо точного ММП считается
условная сумма квадратов, и перебор ускоряется в разы. Финальная выбранная
модель потом переоценивается точным методом. Побочный эффект — на границе
ряды одной и той же длины могут дать разный ответ в зависимости от того,
включилось приближение или нет.

---
# 2. Пишем автоподбор сами

Теперь то, ради чего затевался раздел 1: реализуем алгоритм и посмотрим на
его работу изнутри — какие модели он посещает и в каком порядке.

## 2.1. Шаг 0: выбор $d$

Итеративная процедура «тест → разность → тест» из ноутбука `1.03`.

In [ ]:
def ndiffs_kpss(y, max_d=2, alpha=0.05):
    """Сколько раз дифференцировать, чтобы KPSS перестал отвергать стационарность."""
    z = pd.Series(np.asarray(y, dtype=float)).dropna()
    d = 0
    while d < max_d:
        p_value = kpss(z, regression="c", nlags="auto")[1]
        if p_value >= alpha:          # KPSS не отвергает стационарность — стоп
            break
        z = z.diff().dropna()
        d += 1
    return d


for name, series in [("US consumption", consumption), ("livestock", livestock), ("a10", a10)]:
    print(f"{name:<16} d = {ndiffs_kpss(series)}")

## 2.2. Шаги 1–2: жадный обход

Ключевая деталь реализации — **кэш**. Обход постоянно возвращается к уже
посчитанным моделям (соседи соседа пересекаются), и без кэша половина работы
делалась бы дважды. Заодно кэш даёт нам ровно то, что нужно для разбора:
список всех **действительно оценённых** моделей.

🔍 Как в `statsmodels` включается константа при разном $d$:

| $d$ | Что значит «константа» | Аргумент `trend` |
|---|---|---|
| 0 | среднее уровня ряда $\mu$ | `"c"` |
| 1 | **снос** (drift) — постоянный прирост | `"t"` |
| 2 | не используется вовсе | `"n"` |

Почему при $d = 2$ константы нет: она превратилась бы в квадратичный тренд в
исходных единицах, а такой прогноз почти всегда неправдоподобен (ноутбук `02`,
раздел 6).

In [ ]:
def trend_for(d, with_const):
    """Как обозначить константу/снос в statsmodels при данном d."""
    if not with_const:
        return "n"
    return {0: "c", 1: "t"}.get(d, "n")


def hk_stepwise(y, d=None, max_p=5, max_q=5, max_order=5, verbose=False):
    """Пошаговый автоподбор ARIMA по Хайндману–Кхандакару.

    Возвращает (лучшая модель, d, журнал посещённых моделей).
    Модель кодируется тройкой (p, q, есть ли константа).
    """
    y = np.asarray(y, dtype=float)
    if d is None:
        d = ndiffs_kpss(y)                      # ---- шаг 0

    cache, journal = {}, []

    def score(p, q, c):
        """AICc модели; np.inf, если модель вне сетки или не сошлась."""
        key = (p, q, c)
        if key in cache:
            return cache[key]
        out_of_grid = (
            p < 0 or q < 0 or p > max_p or q > max_q
            or p + q > max_order
            or (d >= 2 and c)                   # при d = 2 константы не бывает
        )
        if out_of_grid:
            cache[key] = np.inf
            return np.inf
        try:
            value = aicc(ARIMA(y, order=(p, d, q), trend=trend_for(d, c)).fit())
        except Exception:
            value = np.inf                      # не сошлась — просто выбываем
        cache[key] = value
        journal.append({"шаг": len(journal), "p": p, "d": d, "q": q,
                        "константа": c, "AICc": value})
        if verbose:
            print(f"  ARIMA({p},{d},{q}){'  + c' if c else '     '}   AICc = {value:9.3f}")
        return value

    with_const = d <= 1                         # ---- шаг 1: четыре стартовые модели
    starts = [(2, 2, with_const), (0, 0, with_const), (1, 0, with_const), (0, 1, with_const)]
    if verbose:
        print("Шаг 1 — стартовый набор:")
    best = min(starts, key=lambda s: score(*s))

    if verbose:
        print(f"лучшая из стартовых: ARIMA({best[0]},{d},{best[1]})\n\nШаг 2 — обход соседей:")

    improved = True                             # ---- шаг 2: спуск по соседям
    while improved:
        improved = False
        p, q, c = best
        neighbours = [
            (p + 1, q, c), (p - 1, q, c),                     # ±1 по p
            (p, q + 1, c), (p, q - 1, c),                     # ±1 по q
            (p + 1, q + 1, c), (p - 1, q - 1, c),             # ±1 по обоим
            (p + 1, q - 1, c), (p - 1, q + 1, c),
            (p, q, not c),                                    # включить/выключить константу
        ]
        for candidate in neighbours:
            if score(*candidate) < score(*best) - 1e-10:
                best = candidate
                improved = True
                break                                          # сразу переходим в новую точку

    return best, d, pd.DataFrame(journal)

## 2.3. Запускаем на US consumption

Тот самый ряд, на котором в ноутбуке `03` пошаговый `AutoARIMA` промахнулся.

In [ ]:
t0 = time.perf_counter()
best_my, d_my, journal = hk_stepwise(consumption, verbose=True)
time_my_stepwise = time.perf_counter() - t0

print(f"\nИтог: ARIMA({best_my[0]}, {d_my}, {best_my[1]})"
      f"{' с константой' if best_my[2] else ' без константы'}")
print(f"оценено моделей: {len(journal)},  время: {time_my_stepwise:.2f} с")

Журнал целиком — это и есть «что автомат видел»:

In [ ]:
journal.round(3)

Нарисуем маршрут по сетке $(p, q)$: каждая точка — оценённая модель, цвет —
AICc, стрелки — переходы между текущими лучшими.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))

finite = journal[np.isfinite(journal["AICc"])]
sc = ax.scatter(finite["q"], finite["p"], c=finite["AICc"], s=260,
                cmap="viridis_r", edgecolor="k", zorder=3)
for _, row in finite.iterrows():
    ax.annotate(f"{row['AICc']:.0f}", (row["q"], row["p"]),
                fontsize=7, ha="center", va="center", color="w", zorder=4)

ax.scatter([best_my[1]], [best_my[0]], s=700, facecolors="none",
           edgecolors="crimson", linewidths=2.5, zorder=5, label="выбранная модель")
ax.set_xlabel("q"); ax.set_ylabel("p")
ax.set_xticks(range(int(finite["q"].max()) + 2))
ax.set_yticks(range(int(finite["p"].max()) + 2))
ax.set_title("Какие точки сетки (p, q) вообще были посещены")
ax.legend(loc="upper right")
fig.colorbar(sc, ax=ax, label="AICc")
plt.show()

print(f"посещено точек сетки: {len(finite)} из {(6 * 6)} возможных при p, q ≤ 5")

🔑 **Вот и вся «магия» автоподбора.** Из 36 клеток сетки алгоритм заглянул в
полтора десятка, и то не подряд, а по цепочке улучшений. Всё, что лежит в
непосещённых клетках, для него не существует.

## 2.4. Сравниваем с полным перебором

Теперь честно переберём всю сетку тем же критерием и посмотрим, совпал ли
ответ.

In [ ]:
t0 = time.perf_counter()
rows = []
for p in range(6):
    for q in range(6):
        if p + q > 5:                       # то же ограничение max_order, что у AutoARIMA
            continue
        for c in (True, False):
            try:
                res = ARIMA(consumption, order=(p, 0, q), trend=trend_for(0, c)).fit()
                value = aicc(res)
            except Exception:
                value = np.inf
            rows.append({"p": p, "q": q, "константа": c, "AICc": value})
time_full_grid = time.perf_counter() - t0

full_grid = pd.DataFrame(rows).sort_values("AICc").reset_index(drop=True)
print(f"полный перебор: {len(full_grid)} моделей за {time_full_grid:.2f} с")
full_grid.head(6).round(3)

In [ ]:
comparison = pd.DataFrame([
    {"стратегия": "пошаговый обход (наш)",
     "модель": f"ARIMA({best_my[0]},{d_my},{best_my[1]})" + (" + c" if best_my[2] else ""),
     "AICc": journal["AICc"].min(),
     "оценено моделей": len(journal),
     "время, с": round(time_my_stepwise, 2)},
    {"стратегия": "полный перебор (наш)",
     "модель": f"ARIMA({full_grid.loc[0, 'p']},0,{full_grid.loc[0, 'q']})"
               + (" + c" if full_grid.loc[0, "константа"] else ""),
     "AICc": full_grid.loc[0, "AICc"],
     "оценено моделей": len(full_grid),
     "время, с": round(time_full_grid, 2)},
])
comparison.round(3)

## 2.5. Сверяемся с библиотекой

Тот же ряд, тот же алгоритм, но реализованный в `statsforecast`. Включим
`trace=True`, чтобы увидеть его собственный журнал.

In [ ]:
y_np = consumption.to_numpy(dtype=np.float64)

t0 = time.perf_counter()
sf_step = AutoARIMA(season_length=1, trace=True).fit(y_np)
time_sf_step = time.perf_counter() - t0

print("\nвыбрано:", arima_string(sf_step.model_).strip())

In [ ]:
t0 = time.perf_counter()
sf_full = AutoARIMA(season_length=1, stepwise=False).fit(y_np)
time_sf_full = time.perf_counter() - t0

print("stepwise=False выбрал:", arima_string(sf_full.model_).strip())

summary = pd.DataFrame([
    {"реализация": "наш обход", "режим": "stepwise",
     "модель": f"ARIMA({best_my[0]},{d_my},{best_my[1]})", "время, с": round(time_my_stepwise, 2)},
    {"реализация": "наш перебор", "режим": "full",
     "модель": f"ARIMA({full_grid.loc[0, 'p']},0,{full_grid.loc[0, 'q']})",
     "время, с": round(time_full_grid, 2)},
    {"реализация": "statsforecast", "режим": "stepwise",
     "модель": arima_string(sf_step.model_).strip(), "время, с": round(time_sf_step, 2)},
    {"реализация": "statsforecast", "режим": "full",
     "модель": arima_string(sf_full.model_).strip(), "время, с": round(time_sf_full, 2)},
])
summary

Разберём, что получилось.

* **Оба полных перебора сошлись на одной модели** — это хорошая новость:
  значит, наша реализация AICc и логика сетки совпадают с библиотечными.
* **Пошаговые версии дали разные ответы.** Наш обход добрался до глобального
  минимума, библиотечный остановился раньше. Это не ошибка ни у кого: у жадного
  спуска результат зависит от **порядка проверки соседей**, от того, принимаем
  ли мы первое улучшение или лучшее из всех, и от мелких различий в оценке
  правдоподобия. Разные реализации одной эвристики — разные локальные минимумы.
* **Время:** пошаговый обход быстрее полного примерно во столько раз, во
  сколько меньше моделей он оценил. На несезонном ряду из 187 точек разница
  в доли секунды, но она умножается на число рядов (в проде их бывают
  миллионы) и на размер сетки (для сезонных моделей она в девять раз больше).

🔍 **Чего в этой таблице делать нельзя — сравнивать AICc между библиотеками.**
`statsmodels` и `statsforecast` по-разному считают число параметров, число
наблюдений после дифференцирования и константу в логарифме правдоподобия.
Числа AICc у них живут в разных «шкалах»: внутри одной библиотеки они
ранжируют модели правильно, а между библиотеками разница может быть
в десятки единиц и не означает ничего. Мы обсуждали этот эффект в ноутбуке
`04`, раздел 4. Поэтому в таблице выше сравниваются **выбранные модели и
время**, но не значения критерия.

## 2.6. Когда пошаговому обходу нельзя верить

Соберём симптомы в одном месте.

| Ситуация | Что делать |
|---|---|
| ряд один и он важный | запустить `stepwise=False` — это дёшево |
| подозрение, что нужен высокий порядок ($p+q>5$) | поднять `max_order` — иначе такие модели вообще недостижимы |
| результат выглядит странно (например, (0,$d$,0) на явно структурированном ряде) | посмотреть `trace=True` и сравнить с ручным перебором |
| рядов миллион, важна скорость | `stepwise=True` + `approximation=True`, и не думать |

🔑 **Практическое правило.** Пошаговый обход экономит время, а не улучшает
качество. Если рядов мало — перебирайте полностью; разница в AICc между
локальным и глобальным минимумом обычно невелика (мы видели: 1.4 единицы,
то есть по правилу 17 из ноутбука `03` — вообще ничего), но узнать это можно
только проверив.

---
# 3. AICc против кросс-валидации

## 3.1. Это два разных вопроса

И `auto.arima`, и мы в разделе 2 выбирали модель по AICc. Но AICc отвечает на
вопрос «насколько хорошо модель описывает **уже наблюдённую историю** с учётом
штрафа за сложность». А пользоваться моделью мы собираемся иначе: нам нужен
**прогноз на $h$ шагов вперёд**. Это разные вещи, и оптимум у них может быть
в разных точках.

Прямой способ измерить то, что нам действительно нужно, — **кросс-валидация
со скользящим началом** (rolling origin): двигать точку отсечки по ряду,
каждый раз обучаться только на прошлом и прогнозировать на $h$ вперёд.

🔍 Обычный `KFold` здесь недопустим: раскидав наблюдения по фолдам случайно,
мы дадим модели увидеть будущее относительно собственного теста. Подробно этот
сюжет разбирался в ноутбуке `Семинары/3/Семинар 1/03` — сегодня мы повторяем
то же сравнение, но для ARIMA.

```
             обучение                      прогноз
  окно 1  ├──────────────┤                 ├─┤
  окно 2  ├────────────────┤               ├─┤
  окно 3  ├──────────────────┤             ├─┤
                                    время →
```

## 3.2. Реализация в десять строк

Напишем скользящую кросс-валидацию для `statsmodels`-моделей.

🔍 Обратите внимание на `refit`: при `refit=True` модель **переобучается
заново** на каждом окне — это честно, но дорого. При `refit=False` параметры
оцениваются один раз, а дальше модель только «доигрывается» вперёд по новым
наблюдениям (метод `.append(..., refit=False)` в `statsmodels`). На практике
используют второе, когда окон много.

In [ ]:
def rolling_cv(y, orders, h=1, min_train=None, trend="c", refit=True):
    """Скользящая кросс-валидация: RMSE прогноза на h шагов для каждой модели.

    orders — словарь {название: (p, d, q)}.
    """
    y = pd.Series(np.asarray(y, dtype=float))
    min_train = min_train or int(len(y) * 0.6)

    errors = {name: [] for name in orders}
    for t in range(min_train, len(y) - h + 1):
        train, actual = y.iloc[:t], y.iloc[t + h - 1]
        for name, order in orders.items():
            try:
                res = ARIMA(train.to_numpy(), order=order, trend=trend).fit()
                pred = float(np.asarray(res.forecast(h))[-1])
            except Exception:
                pred = np.nan
            errors[name].append(actual - pred)

    return pd.Series(
        {name: float(np.sqrt(np.nanmean(np.asarray(e) ** 2))) for name, e in errors.items()},
        name=f"CV-RMSE (h={h})",
    )

## 3.3. Сравниваем ранги

Возьмём шесть кандидатов на US consumption и оценим каждого двумя способами:
по AICc на всей истории и по кросс-валидации с одношаговым горизонтом.

In [ ]:
CANDIDATES = {
    "ARIMA(3,0,0)": (3, 0, 0),
    "ARIMA(1,0,3)": (1, 0, 3),
    "ARIMA(1,0,1)": (1, 0, 1),
    "ARIMA(2,0,2)": (2, 0, 2),
    "ARIMA(1,0,0)": (1, 0, 0),
    "ARIMA(0,0,0)": (0, 0, 0),
}

aicc_scores = pd.Series(
    {name: aicc(ARIMA(consumption, order=order, trend="c").fit())
     for name, order in CANDIDATES.items()},
    name="AICc",
)

t0 = time.perf_counter()
cv_scores = rolling_cv(consumption, CANDIDATES, h=1, min_train=120, trend="c")
time_cv = time.perf_counter() - t0

ranking = pd.concat([aicc_scores, cv_scores], axis=1)
ranking["ранг AICc"] = ranking["AICc"].rank().astype(int)
ranking["ранг CV"] = ranking[cv_scores.name].rank().astype(int)
ranking = ranking.sort_values("ранг AICc")
print(f"кросс-валидация: {len(CANDIDATES)} моделей × ~{187 - 120} окон, {time_cv:.1f} с")
ranking.round(4)

## 3.4. Что здесь важно увидеть

Два столбца рангов — про разные вещи, и совпадать они не обязаны. Посмотрите
на них внимательно и обратите внимание на три момента.

**Первое: расхождение рангов — норма, а не сбой.** AICc усредняет качество
описания по всей истории, CV измеряет ошибку конкретно на горизонте $h$.
Даже когда лидер совпадает, порядок остальных моделей обычно другой.

**Второе: разброс CV-RMSE между кандидатами крошечный.** Разница между лучшей
и худшей осмысленной моделью — единицы процентов, а между соседями по
таблице — доли процента. Это тот же эффект, что мы видели в ноутбуке `03`,
раздел 8: несколько разных моделей описывают ряд одинаково хорошо.

**Третье: цена.** AICc — это $k$ подгонок (по одной на модель). CV — это
$k \times (\text{число окон})$ подгонок. На нашем игрушечном примере разница
в секунды, на панели из тысяч рядов — в часы.

In [ ]:
print(f"AICc:  {len(CANDIDATES)} подгонок")
print(f"CV:    {len(CANDIDATES)} × {187 - 120} = {len(CANDIDATES) * (187 - 120)} подгонок, {time_cv:.1f} с")
print(f"\nразмах CV-RMSE: от {cv_scores.min():.4f} до {cv_scores.max():.4f} "
      f"({(cv_scores.max() / cv_scores.min() - 1):.1%} разницы)")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(ranking["ранг AICc"], ranking["ранг CV"], s=90, zorder=3)
for name, row in ranking.iterrows():
    ax.annotate(name.replace("ARIMA", ""), (row["ранг AICc"], row["ранг CV"]),
                textcoords="offset points", xytext=(8, 4), fontsize=8)
lim = [0.5, len(ranking) + 0.5]
ax.plot(lim, lim, ls="--", color="0.6", label="полное согласие")
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("ранг по AICc"); ax.set_ylabel("ранг по CV-RMSE")
ax.set_title("Если бы критерии были согласны, точки легли бы на диагональ")
ax.legend()
plt.show()

🔑 **Как выбирать на практике.**

* **Мало рядов, важен результат** — кросс-валидация на том горизонте, который
  вам реально нужен. Она оптимизирует именно вашу задачу.
* **Много рядов или нужен быстрый ответ** — AICc. Он бесплатный и в среднем
  не сильно хуже.
* **Никогда** не выбирайте модель по ошибке на отложенном тесте, который потом
  же используете как оценку качества: так вы подгоните модель под тест и
  получите завышенную оценку. Выбор — по train (AICc или внутренняя CV), тест —
  один раз в самом конце.

🔍 И ещё одна тонкость про CV: результат зависит от числа окон и от их длины.
На коротком ряду с пятью окнами победитель может смениться от того, что в
одном окне случился выброс. Кросс-валидация — тоже оценка со своей
дисперсией, а не истина.

---
# 4. Когда все кандидаты проваливают тесты

## 4.1. Постановка

Правило 16 из ноутбука `03` говорит: остатки хорошей модели должны быть
неотличимы от белого шума, а если Ljung–Box отвергает — возвращайтесь к
шагу 4. Что делать, если **ни одна** модель из доступных проверку не проходит?

Ответ Хайндмана категоричен: **берите лучшую доступную модель и прогнозируйте.**
Провал теста означает, что в данных осталась структура, которую эта модель не
описывает, — но не означает, что её прогноз хуже, чем ничего.

Смоделируем ситуацию буквально. Возьмём ряд `a10` — он сильно сезонный, — а
подбирать будем только **несезонные** модели. Сезонная ARIMA будет в семинаре 5;
сегодня у нас её нет, и это как раз тот случай, когда инструмент заведомо
недостаточен.

In [ ]:
tsdisplay(a10, lags=48, title="a10: расходы на препараты, месячные данные")
plt.show()

Всплески ACF на лагах 12, 24, 36 — классическая сезонная подпись. Несезонная
модель их описать не может: у неё просто нет параметров, которые связывали бы
$y_t$ с $y_{t-12}$, не потратив на это двенадцать AR-коэффициентов.

## 4.2. Все кандидаты валят Ljung–Box

In [ ]:
CAND_A10 = [(0, 1, 1), (1, 1, 1), (2, 1, 2), (3, 1, 2), (5, 1, 0), (2, 1, 3)]

rows = []
fits_a10 = {}
for order in CAND_A10:
    res = ARIMA(a10, order=order, trend="t").fit()
    fits_a10[order] = res
    lb = acorr_ljungbox(res.resid[order[0] + 1:], lags=[24],
                        model_df=order[0] + order[2]).iloc[0]
    rows.append({"модель": f"ARIMA{order}", "AICc": aicc(res),
                 "Ljung-Box Q (лаг 24)": lb["lb_stat"],
                 "p-value": lb["lb_pvalue"]})

table_a10 = pd.DataFrame(rows).sort_values("AICc").reset_index(drop=True)
table_a10.round(4)

p-value везде исчезающе мал — порядка $10^{-39}$. Это не «на грани», это
полный, абсолютный провал: остатки очевидно автокоррелированы, и никакое
увеличение $p$ или $q$ в разумных пределах не спасает.

Посмотрим на ACF остатков лучшей из них — там прекрасно видно, что именно
осталось необъяснённым.

In [ ]:
best_a10_order = CAND_A10[int(np.argmin([aicc(fits_a10[o]) for o in CAND_A10]))]
resid_best = pd.Series(np.asarray(fits_a10[best_a10_order].resid)[best_a10_order[0] + 1:])

fig, ax = plt.subplots(figsize=(12, 3.2))
plot_acf(resid_best, lags=48, ax=ax, title=f"ACF остатков ARIMA{best_a10_order}: сезонность осталась")
plt.show()

Столбики на лагах 12, 24, 36 никуда не делись — модель забрала тренд и
краткосрочную динамику, а годовой цикл остался целиком в остатках.

## 4.3. И тем не менее прогноз полезен

Отложим последний год и сравним «провалившую все тесты» модель с двумя
бенчмарками.

In [ ]:
H = 12
train_a10, test_a10 = a10.iloc[:-H], a10.iloc[-H:]

res_a10 = ARIMA(train_a10, order=best_a10_order, trend="t").fit()
fc_a10 = np.asarray(res_a10.forecast(H))

preds = {
    f"ARIMA{best_a10_order} (валит Ljung-Box)": fc_a10,
    "наивный (последнее значение)": np.repeat(train_a10.iloc[-1], H),
    "среднее train": np.repeat(train_a10.mean(), H),
    "сезонно-наивный (год назад)": train_a10.iloc[-12:].to_numpy(),
}

scores_a10 = pd.DataFrame({
    name: {"RMSE": float(np.sqrt(np.mean((test_a10.to_numpy() - p) ** 2))),
           "MAE": float(np.mean(np.abs(test_a10.to_numpy() - p)))}
    for name, p in preds.items()
}).T.sort_values("RMSE")
scores_a10.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
train_a10.iloc[-36:].plot(ax=ax, color="0.4", label="train")
test_a10.plot(ax=ax, color="k", lw=2, label="факт (test)")
for name, p in preds.items():
    ax.plot(test_a10.index, p, ls="--", lw=1.6, label=name)
ax.set_title("a10: несезонная ARIMA против бенчмарков")
ax.legend(fontsize=8)
plt.show()

Что видно на графике и в таблице:

* несезонная ARIMA **не воспроизводит сезонную волну** — она и не может;
* но тренд она ловит, и наивный прогноз с прогнозом «по среднему» обходит;
* **сезонно-наивный** прогноз (взять значения год назад) на этом ряде
  оказывается сильным конкурентом именно потому, что содержит то, чего нет у
  нашей модели, — сезонность.

🔑 **Мораль, которую стоит запомнить дословно (слайд 179 лекции).**
«Не проходит Ljung–Box» — это диагноз, а не запрет на прогнозирование. Провал
теста говорит две вещи:

1. **точечный прогноз можно улучшить** — в остатках осталась предсказуемая
   структура (здесь — сезонность; лечится сезонной моделью в семинаре 5);
2. **интервалам верить нельзя** — формулы из ноутбука `02` выведены в
   предположении некоррелированных ошибок, а они коррелированы, значит
   настоящая неопределённость больше расчётной.

Но пока лучшей модели нет — прогнозируйте имеющейся, честно оговорив
ограничения. Отказ от прогноза — это тоже прогноз, просто худший.

---
# Итоги

* **`auto.arima` отвечает на три вопроса разными способами:** $d$ — тестом на
  единичный корень, $(p, q)$ и константа — минимумом AICc. Смешивать их
  нельзя, потому что AICc не сравнивает модели с разным $d$ (раздел 1.1).
* **Алгоритм Хайндмана–Кхандакара — жадный спуск:** четыре стартовые модели,
  потом шаги $\pm 1$ по $p$, $q$ и переключение константы, пока AICc падает
  (раздел 1.3). Мы реализовали его в сорок строк (раздел 2.2).
* **Пошаговый обход не гарантирует глобального минимума** и посещает лишь
  малую часть сетки. Разные реализации одной эвристики приходят в разные
  локальные минимумы (раздел 2.5). Если ряд один и он важен — перебирайте
  полностью, это дёшево.
* **`max_order=5` ограничивает сумму $p+q$**, а не каждый порядок по
  отдельности: ARIMA(4,$d$,4) недостижима при настройках по умолчанию даже
  при `stepwise=False` (раздел 1.5).
* **AICc и кросс-валидация — разные критерии** и дают разные ранжирования.
  AICc почти бесплатен, CV измеряет именно то, что нужно, но стоит
  «число моделей × число окон» подгонок (раздел 3).
* **Провал Ljung–Box — не запрет на прогноз.** Он означает «точечный прогноз
  можно улучшить» и «интервалам верить нельзя», но лучшая доступная модель
  всё равно обычно бьёт наивные бенчмарки (раздел 4).

